In [1]:
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)
from peft import LoraConfig, get_peft_model, TaskType

/home/simonr/Coding/OntologyScraper/OntologyScraperEnv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("/home/simonr/Coding/OntologyScraper/alc_english_pairs_10000.csv")
dataset = Dataset.from_pandas(df).train_test_split(test_size=0.1)

In [3]:
MODEL_NAME = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

Loading weights: 100%|███████████████████████████████████| 131/131 [00:00<00:00, 17187.08it/s]


In [4]:
alc_symbols = ["⊑", "⊓", "⊔", "¬", "∃", "∀"]
tokenizer.add_tokens(alc_symbols)

model.resize_token_embeddings(len(tokenizer))

Embedding(32106, 512)

In [5]:
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, 
    r=8, 
    lora_alpha=32,       
    lora_dropout=0.1,
    target_modules=["q", "v"]
)
model = get_peft_model(model, peft_config)


In [6]:
PREFIX = "parse logic: "

def preprocess_function(examples):
    inputs = [PREFIX + doc for doc in examples["english_sentence"]]
    model_inputs = tokenizer(inputs, max_length=256, truncation=True)

    labels = tokenizer(text_target=examples["alc_statement"], max_length=256, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

Map: 100%|██████████████████████████████████████| 1000/1000 [00:00<00:00, 54393.78 examples/s]


In [7]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-small-lora-logic",
    eval_strategy="epoch",
    learning_rate=1e-3, # LoRA verträgt/braucht oft eine etwas höhere Lernrate (z. B. 1e-3)
    per_device_train_batch_size=16,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=False, # Auf True setzen, wenn eine passende GPU genutzt wird
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,2.490916,1.023646
2,1.096010,0.944002
3,1.011357,0.914798
4,0.973688,0.898206
5,0.952800,0.888750


/home/simonr/Coding/OntologyScraper/OntologyScraperEnv/lib/python3.14/site-packages/peft/utils/save_and_load.py:452: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/home/simonr/Coding/OntologyScraper/OntologyScraperEnv/lib/python3.14/site-packages/peft/utils/save_and_load.py:452: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/home/simonr/Coding/OntologyScraper/OntologyScraperEnv/lib/python3.14/site-packages/peft/utils/save_and_load.py:452: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/home/simonr/Coding/OntologyScraper/OntologyScraperEnv/lib/python3.14/site-packages/peft/utils/save_and_load.py:452: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/home/si

TrainOutput(global_step=2815, training_loss=1.26476168319133, metrics={'train_runtime': 125.0652, 'train_samples_per_second': 359.812, 'train_steps_per_second': 22.508, 'total_flos': 262440882536448.0, 'train_loss': 1.26476168319133, 'epoch': 5.0})

In [8]:
model.save_pretrained("./t5_fineTuned")
tokenizer.save_pretrained("./t5_fineTuned")

/home/simonr/Coding/OntologyScraper/OntologyScraperEnv/lib/python3.14/site-packages/peft/utils/save_and_load.py:452: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


('./t5_fineTuned/tokenizer_config.json', './t5_fineTuned/tokenizer.json')